# Model Performance Report

This notebook is the display and export surface for the final model-performance table. It uses reusable functions from `src/`, applies the shared audit/QC/CV contract, displays all intermediate evidence tables, and writes one consolidated CSV: `results/model_performance_summary.csv`.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import DEFAULT_CONFIG, set_global_seeds
from src.data.audit import add_visit_time, analysis_population_counts, audit_visit_patterns, visit_pattern_table
from src.data.missingness import feature_missingness_report, feature_visit_site_missingness_matrix
from src.data.mri_qc import harmonisation_leakage_policy, mri_outlier_table, plot_feature_distributions_by_site, site_effect_screen
from src.data.trackfa import feature_catalog, raw_direct_combination_audit
from src.data.trackfa_pairs import infer_trackfa_feature_groups
from src.eval.clinical_validity import clinical_validity
from src.eval.intervals import interval_effect_summary
from src.eval.single_feature import single_feature_interval_baselines
from src.eval.stability import selected_feature_jaccard
from src.models.srm_global import srm_global_loocv, srm_global_repeated_group_cv
from src.reporting.model_performance import (
    append_log_model_summaries,
    assemble_performance_rows,
    best_model_rows_from_logs,
    cv_contract_table,
    save_one_performance_csv,
)

set_global_seeds(DEFAULT_CONFIG.random_state)
RANDOM_SEED = DEFAULT_CONFIG.random_state
CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits
N_BOOT = 300

long_path = REPO_ROOT / "data" / "processed" / "trackfa_long.csv"
pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not pairs_path.exists():
    raise FileNotFoundError(f"Required modelling dataset not found: {pairs_path}")

long_df = add_visit_time(pd.read_csv(long_path), visit_col="visit")
pairs_df = pd.read_csv(pairs_path)
feature_groups = infer_trackfa_feature_groups(pairs_df)
cat = feature_catalog(long_df, pairs_df)
imaging_cols = [c for c in feature_groups.all_neuroimaging if c in long_df.columns]
if not imaging_cols:
    imaging_cols = cat["long_imaging_columns"]

print(f"Loaded {long_path.name}: {long_df.shape[0]} rows, {long_df['subject_id'].nunique()} subjects")
print(f"Loaded {pairs_path.name}: {pairs_df.shape[0]} rows")
print(f"MRI features available for reporting: {len(imaging_cols)}")


DISPLAY_COLUMN_NAMES = {
    "source": "Source",
    "raw_rows": "Raw Rows",
    "raw_columns": "Raw Columns",
    "visit_rows_used_for_direct_join": "Visit Rows Used for Direct Join",
    "unique_participants": "Unique Participants",
    "source_presence": "Source Presence",
    "n_rows": "Number of Rows",
    "visit": "Visit",
    "source_block": "Source Block",
    "n_columns": "Number of Columns",
    "mean_missing_pct": "Mean Missing %",
    "max_missing_pct": "Maximum Missing %",
    "column": "Column",
    "missing_pct": "Missing %",
    "n_missing": "Number Missing",
}

DISPLAY_VALUE_REPLACEMENTS = {
    "both": "Clinical and Imaging",
    "left_only": "Clinical Only",
    "right_only": "Imaging Only",
    "clinical_raw": "Raw Clinical Columns",
    "imaging_raw": "Raw Imaging Columns",
}


def display_table(frame: pd.DataFrame):
    """Display audit tables with full labels and percentage units."""
    out = frame.copy()
    for col in out.columns:
        if col.endswith("pct") or col.endswith("_pct") or col in {"missing_pct", "mean_missing_pct", "max_missing_pct"}:
            out[col] = pd.to_numeric(out[col], errors="coerce") * 100.0
    out = out.replace(DISPLAY_VALUE_REPLACEMENTS)
    out = out.rename(columns={k: v for k, v in DISPLAY_COLUMN_NAMES.items() if k in out.columns})
    percent_cols = [c for c in out.columns if c.endswith("%")]
    if percent_cols:
        display(out.style.format({c: "{:.2f}%" for c in percent_cols}, na_rep=""))
    else:
        display(out)



def keep_clinical_total_missing_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """For raw clinical columns, display totals plus core demographics/genetics."""
    if "source_block" not in frame.columns or "column" not in frame.columns:
        return frame
    source = frame["source_block"].astype(str)
    column = frame["column"].astype(str)
    is_raw_clinical = source.eq("clinical_raw") | column.str.startswith("clinical__")
    clinical_name = column.str.replace(r"^clinical__", "", regex=True)
    core_clinical = {"age", "gender", "gaa_1"}
    keep_clinical = clinical_name.str.endswith("_total") | clinical_name.isin(core_clinical)
    keep = ~is_raw_clinical | keep_clinical
    return frame.loc[keep].copy()


Loaded trackfa_long.csv: 522 rows, 174 subjects
Loaded trackfa_pairs_drop3poms.csv: 207 rows
MRI features available for reporting: 146


## Shared Audit, QC, and CV Contract


In [2]:
cv_contract = cv_contract_table(
    outer_cv=f"subject-level grouped {CV_N_SPLITS}-fold CV; repeated wrapper available for final sensitivity checks",
    inner_cv="grouped subject-level CV inside outer training subjects for hyperparameter/feature-selection tuning",
    grouping_unit="subject_id for subject-level visit data; participant group for pair-table interval data",
)
display(cv_contract)


,component,requirement
0,Data audit,"Visit pattern, analysis population, and Featur..."
1,MRI QC,"Distribution by site, site-effect screen, outl..."
2,Outer CV,subject-level grouped 5-fold CV; repeated wrap...
3,Inner CV,grouped subject-level CV inside outer training...
4,Grouping,All visits/intervals for one participant stay ...
5,Leakage policy,"Imputation, scaling, feature selection, harmon..."


## Raw Direct Combination Missingness

Before the cleaned analytic audit, this section directly combines raw REDCap visit rows and raw imaging MasterFile visit rows by `participant_id` and `visit`. This reports original source-level missingness before filtering, feature selection, pair construction, or complete-imaging restrictions.


In [3]:
raw_direct_audit = raw_direct_combination_audit()
print("Raw source sizes and direct-join size")
display_table(raw_direct_audit["source_summary"])
print("Direct outer-join source presence")
display_table(raw_direct_audit["presence_summary"])
print("Direct outer-join source presence by visit")
display_table(raw_direct_audit["visit_presence"])
print("Missingness summary by raw source block")
display_table(raw_direct_audit["block_summary"])
print("Top missing columns after direct raw combination")
display_table(keep_clinical_total_missing_columns(raw_direct_audit["missingness"]).head(40))


Raw source sizes and direct-join size


,Source,Raw Rows,Raw Columns,Visit Rows Used for Direct Join,Unique Participants
0,REDCap raw export,1076,483,807,269
1,Imaging MasterFile all sheets,793,157,747,269
2,Direct outer join,807,639,807,269


Direct outer-join source presence


,Source Presence,Number of Rows
0,Clinical and Imaging,747
1,Clinical Only,60
2,Imaging Only,0


Direct outer-join source presence by visit


,Visit,Source Presence,Number of Rows
0,1,Clinical Only,1
1,1,Imaging Only,0
2,1,Clinical and Imaging,268
3,2,Clinical Only,20
4,2,Imaging Only,0
5,2,Clinical and Imaging,249
6,3,Clinical Only,39
7,3,Imaging Only,0
8,3,Clinical and Imaging,230


Missingness summary by raw source block


,Source Block,Number of Columns,Mean Missing %,Maximum Missing %
0,Raw Clinical Columns,481,80.55%,100.00%
1,Raw Imaging Columns,155,20.65%,39.65%


Top missing columns after direct raw combination


,Column,Missing %,Source Block
0,clinical__age,100.00%,Raw Clinical Columns
8,clinical__gaa_1,100.00%,Raw Clinical Columns
10,clinical__gender,100.00%,Raw Clinical Columns
476,clinical__upenn_fxn_total,66.67%,Raw Clinical Columns
480,imaging__tNAA_myo_Ins,39.65%,Raw Imaging Columns
481,imaging__DN_suscept,36.93%,Raw Imaging Columns
482,imaging__DN_vol,36.93%,Raw Imaging Columns
483,imaging__AD_CST,23.42%,Raw Imaging Columns
484,imaging__AD_ICP,23.42%,Raw Imaging Columns
485,imaging__AD_MCP,23.42%,Raw Imaging Columns


In [4]:
visit_audit = audit_visit_patterns(long_df, subject_col="subject_id", visit_col="visit")
pattern_table = visit_pattern_table(visit_audit)
population_table = analysis_population_counts(long_df, subject_col="subject_id", visit_col="visit")

print("Visit pattern report")
display(pattern_table)
print("Analysis populations")
display(population_table)


Visit pattern report


,pattern,meaning,n_subjects,percent_subjects
0,111,"V1, V2, and V3 available",174,100.0
1,110,V1 and V2 available; V3 missing,0,0.0
2,101,V1 and V3 available; V2 missing,0,0.0
3,011,V2 and V3 available; V1 missing,0,0.0
4,100,Only V1 available,0,0.0
5,010,Only V2 available,0,0.0
6,001,Only V3 available,0,0.0


Analysis populations


,population,required_visits,n_subjects,role
0,V1-V3 primary cohort,"V1,V3",174,Primary 24-month annualised paired change
1,V1-V2 cohort,"V1,V2",174,Secondary 12-month interval
2,V2-V3 cohort,"V2,V3",174,Secondary 12-month interval
3,Complete V1-V2-V3 cohort,"V1,V2,V3",174,Consistency and trajectory checks
4,All longitudinal with any adjacent pair,"V1,V2 or V2,V3",174,Maximum adjacent-interval sensitivity check


In [5]:
missingness_report = feature_missingness_report(long_df, imaging_cols, by=("visit", "site"), concentration_spread=0.25)
missingness_matrix = feature_visit_site_missingness_matrix(long_df, imaging_cols, visit_col="visit", site_col="site")

print("Global MRI missingness")
display(missingness_report["global"].head(30))
print("Feature x Visit x Site missingness matrix")
display(missingness_matrix["matrix"])
print("Missingness concentration flags")
display(missingness_report["flags"].head(30))


Global MRI missingness


,feature,missing_pct
0,AD_ACR,0.220307
1,AD_ALIC,0.220307
2,AD_CP,0.220307
3,AD_CST,0.220307
4,AD_Cing,0.220307
5,AD_Cing_h,0.220307
6,AD_EC,0.220307
7,AD_Fx,0.220307
8,AD_Fx_ST,0.220307
9,AD_ICP,0.220307


Feature x Visit x Site missingness matrix


visit_site,V1|1.0,V1|2.0,V1|3.0,V1|5.0,V1|6.0,V1|7.0,V2|1.0,V2|2.0,V2|3.0,V2|5.0,V2|6.0,V2|7.0,V3|1.0,V3|2.0,V3|3.0,V3|5.0,V3|6.0,V3|7.0
feature,,,,,,,,,,,,,,,,,,
AD_ACR,0.083333,0.204545,0.20,0.200000,0.043478,0.0,0.125000,0.386364,0.25,0.218182,0.043478,0.000,0.166667,0.409091,0.35,0.254545,0.304348,0.000
AD_ALIC,0.083333,0.204545,0.20,0.200000,0.043478,0.0,0.125000,0.386364,0.25,0.218182,0.043478,0.000,0.166667,0.409091,0.35,0.254545,0.304348,0.000
AD_CP,0.083333,0.204545,0.20,0.200000,0.043478,0.0,0.125000,0.386364,0.25,0.218182,0.043478,0.000,0.166667,0.409091,0.35,0.254545,0.304348,0.000
AD_CST,0.083333,0.204545,0.20,0.200000,0.043478,0.0,0.125000,0.386364,0.25,0.218182,0.043478,0.000,0.166667,0.409091,0.35,0.254545,0.304348,0.000
AD_Cing,0.083333,0.204545,0.20,0.200000,0.043478,0.0,0.125000,0.386364,0.25,0.218182,0.043478,0.000,0.166667,0.409091,0.35,0.254545,0.304348,0.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
sFA_c3c5,0.041667,0.272727,0.20,0.181818,0.000000,0.0,0.125000,0.409091,0.25,0.181818,0.000000,0.000,0.166667,0.454545,0.30,0.272727,0.260870,0.000
sMD_c3c5,0.041667,0.272727,0.20,0.181818,0.000000,0.0,0.125000,0.409091,0.25,0.181818,0.000000,0.000,0.166667,0.454545,0.30,0.272727,0.260870,0.000
sRD_c3c5,0.041667,0.272727,0.20,0.181818,0.000000,0.0,0.125000,0.409091,0.25,0.181818,0.000000,0.000,0.166667,0.454545,0.30,0.272727,0.260870,0.000


Missingness concentration flags


,feature,reason,max_missing_pct,min_missing_pct
0,AD_ACR,missingness spread >= 0.25 in by_site,0.333333,0.0
1,AD_ALIC,missingness spread >= 0.25 in by_site,0.333333,0.0
2,AD_CP,missingness spread >= 0.25 in by_site,0.333333,0.0
3,AD_CST,missingness spread >= 0.25 in by_site,0.333333,0.0
4,AD_Cing,missingness spread >= 0.25 in by_site,0.333333,0.0
5,AD_Cing_h,missingness spread >= 0.25 in by_site,0.333333,0.0
6,AD_EC,missingness spread >= 0.25 in by_site,0.333333,0.0
7,AD_Fx,missingness spread >= 0.25 in by_site,0.333333,0.0
8,AD_Fx_ST,missingness spread >= 0.25 in by_site,0.333333,0.0
9,AD_ICP,missingness spread >= 0.25 in by_site,0.333333,0.0


In [6]:
print("QC 1: MRI feature distributions by site")
display(plot_feature_distributions_by_site(long_df, imaging_cols, site_col="site", max_features=12))

print("QC 2: Feature ~ Site + Age + Sex + DiseaseSeverity")
site_effects = site_effect_screen(long_df, imaging_cols, site_col="site")
display(site_effects.head(30))

print("Outlier review table: flags for review, not automatic deletion")
outlier_review = mri_outlier_table(long_df, imaging_cols, group_cols=("visit", "site"))
display(outlier_review.head(30))

print("QC 3: harmonisation leakage policy")
display(harmonisation_leakage_policy())


QC 1: MRI feature distributions by site


<Figure size 1500x1280 with 12 Axes>

QC 2: Feature ~ Site + Age + Sex + DiseaseSeverity


,feature,n,site_levels,site_r2_delta,site_f,site_p_value,covariates_used
0,AD_gCC,406,6,0.354947,230.783940,1.711480e-41,"age,gender,mfars_total"
1,AD_ILF_IFOF,406,6,0.333896,218.980564,7.634393e-40,"age,gender,mfars_total"
2,AD_bCC,406,6,0.329053,217.870386,1.095325e-39,"age,gender,mfars_total"
3,AD_sCC,406,6,0.331950,211.045200,1.022428e-38,"age,gender,mfars_total"
4,AD_Fx,406,6,0.338124,205.972086,5.467924e-38,"age,gender,mfars_total"
5,AD_RLIC,406,6,0.310314,192.778823,4.579939e-36,"age,gender,mfars_total"
6,AD_SLF,406,6,0.296420,179.654694,4.143461e-34,"age,gender,mfars_total"
7,MD_gCC,406,6,0.299375,178.344387,6.533758e-34,"age,gender,mfars_total"
8,AD_ICP,406,6,0.290658,169.353365,1.530487e-32,"age,gender,mfars_total"
9,AD_CP,406,6,0.259879,159.545489,5.061291e-31,"age,gender,mfars_total"


Outlier review table: flags for review, not automatic deletion


,visit,site,feature,n,n_outliers,outlier_pct,lower_fence,upper_fence,min_flagged,max_flagged
0,2,1.0,MD_Cing_h,21,4,0.190476,0.000740,0.000793,0.000714,0.000829
1,1,1.0,MD_PCT,22,3,0.136364,0.000499,0.000949,0.000997,0.001076
2,2,2.0,RD_gCC,27,3,0.111111,0.000297,0.000520,0.000531,0.000568
3,2,5.0,MD_Fx,43,3,0.069767,0.000461,0.001368,0.001375,0.001447
4,3,5.0,RD_PCR,41,3,0.073171,0.000480,0.000590,0.000462,0.000473
5,1,1.0,AD_PCT,22,2,0.090909,0.000733,0.001281,0.001379,0.001399
6,1,1.0,RD_PCT,22,2,0.090909,0.000274,0.000891,0.000895,0.000925
7,1,2.0,RD_SCR,35,2,0.057143,0.000386,0.000567,0.000577,0.000603
8,1,3.0,MD_gCC,16,2,0.125000,0.000646,0.000760,0.000784,0.000797
9,1,3.0,RD_gCC,16,2,0.125000,0.000291,0.000543,0.000547,0.000555


QC 3: harmonisation leakage policy


,rule,reason
0,Estimate harmonisation parameters inside each ...,Using held-out subjects to estimate site/scann...
1,Apply learned parameters unchanged to validati...,The outer test fold must remain untouched unti...
2,Preserve within-subject longitudinal change.,The biomarker objective is progression sensiti...
3,Report pre/post site-effect diagnostics if Com...,A site correction is only defensible if it red...


## Model Performance Table

The SRM Global Linear model is recomputed here on the subject-level longitudinal table for the primary and secondary intervals. Other model families are appended from existing optimization logs using the same required question/metric schema; unavailable metrics remain explicitly marked as unavailable rather than inferred.


In [7]:
srm_interval_results = {}
for interval_name, start_visit, end_visit in [
    ("V1->V3", 1, 3),
    ("V1->V2", 1, 2),
    ("V2->V3", 2, 3),
]:
    srm_interval_results[interval_name] = srm_global_loocv(
        long_df,
        imaging_cols,
        subject_col="subject_id",
        visit_col="visit",
        selection_method="none",
        k=8,
        ridge=1e-6,
        covariance_shrinkage=0.45,
        z_clip=None,
        cv_n_splits=CV_N_SPLITS,
        random_seed=RANDOM_SEED,
        compute_ci=False,
        start_visit=start_visit,
        end_visit=end_visit,
    )

composite_interval_rows = []
for interval_name, result in srm_interval_results.items():
    summary = interval_effect_summary(
        result["oof_df"],
        subject_col="subject_id",
        visit_col="visit",
        score_col="score",
        time_col="time_years",
        intervals=[(result["start_visit"], result["end_visit"], interval_name, interval_name != "V1->V3")],
        n_boot=N_BOOT,
        seed=RANDOM_SEED,
    )
    composite_interval_rows.append(summary)
composite_intervals = pd.concat(composite_interval_rows, ignore_index=True)
print("Composite interval performance")
display(composite_intervals)


Composite interval performance


,interval,start_visit,end_visit,annualised,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive
0,V1->V3,1,3,False,101,6.072755,4.577050,1.326784,1.060792,1.619696,0.920792
1,V1->V2,1,2,True,108,4.106699,3.421995,1.200089,1.001361,1.434438,0.879630
2,V2->V3,2,3,True,100,1.259162,2.933010,0.429307,0.261144,0.640850,0.670000


In [8]:
single_feature_intervals = single_feature_interval_baselines(
    long_df,
    imaging_cols,
    subject_col="subject_id",
    visit_col="visit",
    time_col="time_years",
    n_boot=N_BOOT,
    seed=RANDOM_SEED,
)
print("Strongest individual MRI features by interval")
display(single_feature_intervals.groupby("interval", group_keys=False).head(5))

clinical_vars = [c for c in ["FARS", "SARA", "mfars_total", "sara_total"] if c in long_df.columns]
clinical_interval_parts = []
for clinical_col in clinical_vars:
    clinical_interval_parts.append(
        interval_effect_summary(
            long_df.dropna(subset=[clinical_col]),
            subject_col="subject_id",
            visit_col="visit",
            score_col=clinical_col,
            time_col="time_years",
            n_boot=N_BOOT,
            seed=RANDOM_SEED,
        ).assign(feature=clinical_col, kind="clinical_scale")
    )
clinical_intervals = pd.concat(clinical_interval_parts, ignore_index=True) if clinical_interval_parts else pd.DataFrame()
print("Clinical-scale benchmarks")
display(clinical_intervals)


Strongest individual MRI features by interval


,interval,start_visit,end_visit,annualised,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive,feature,kind
0,V1->V2,V1,V2,True,149,-1224.194644,1857.318129,-0.659120,-0.886112,-0.437390,0.214765,Cerebellum_Cortex_CerebNet,single_mri_feature
1,V1->V2,V1,V2,True,149,-1347.653107,2112.606208,-0.637910,-0.876610,-0.418949,0.214765,Cerebellum_CerebNet,single_mri_feature
2,V1->V2,V1,V2,True,149,-1535.215013,2437.834877,-0.629745,-0.879234,-0.411334,0.201342,cerebellumFS,single_mri_feature
3,V1->V2,V1,V2,True,149,-1251.871389,2023.449903,-0.618682,-0.859347,-0.399685,0.234899,Cerebellum_Cortex_FS,single_mri_feature
4,V1->V2,V1,V2,True,149,-191.253947,338.599147,-0.564839,-0.792716,-0.388927,0.268456,Whole_brainstem,single_mri_feature
146,V1->V3,V1,V3,False,134,-2795.656343,2862.038521,-0.976806,-1.167725,-0.831867,0.164179,cerebellumFS,single_mri_feature
147,V1->V3,V1,V3,False,134,-2192.912799,2250.074356,-0.974596,-1.147427,-0.811102,0.141791,Cerebellum_Cortex_FS,single_mri_feature
148,V1->V3,V1,V3,False,134,-2409.456649,2536.964178,-0.949740,-1.154683,-0.790027,0.164179,Cerebellum_CerebNet,single_mri_feature
149,V1->V3,V1,V3,False,134,-2091.460799,2218.296284,-0.942823,-1.133571,-0.794404,0.134328,Cerebellum_Cortex_CerebNet,single_mri_feature
150,V1->V3,V1,V3,False,135,-2362.170370,2916.727256,-0.809870,-1.050714,-0.644354,0.140741,Cereb_vol,single_mri_feature


Clinical-scale benchmarks


,interval,start_visit,end_visit,annualised,n_pairs,mean_change,sd_change,d_z,d_z_ci_low,d_z_ci_high,p_delta_positive,feature,kind
0,V1->V3,V1,V3,False,150,5.271111,8.010399,0.658033,0.527607,0.802225,0.733333,mfars_total,clinical_scale
1,V1->V2,V1,V2,True,165,3.032323,5.592166,0.542245,0.407338,0.694389,0.660606,mfars_total,clinical_scale
2,V2->V3,V2,V3,True,149,2.204698,5.800426,0.380092,0.256318,0.556495,0.637584,mfars_total,clinical_scale
3,V1->V3,V1,V3,False,150,2.543333,3.491090,0.728521,0.566647,0.895133,0.733333,sara_total,clinical_scale
4,V1->V2,V1,V2,True,164,1.387195,2.604459,0.532623,0.411083,0.675898,0.658537,sara_total,clinical_scale
5,V2->V3,V2,V3,True,150,1.216667,2.983971,0.407734,0.246863,0.578295,0.620000,sara_total,clinical_scale


In [9]:
primary_oof = srm_interval_results["V1->V3"]["oof_df"]
clinical_validity_table = clinical_validity(
    primary_oof,
    long_df,
    subject_col="subject_id",
    visit_col="visit",
    score_col="score",
    clinical_variables=clinical_vars,
    start_visit="V1",
    end_visit="V3",
) if clinical_vars else pd.DataFrame()
print("Clinical validation of locked OOF composite scores")
display(clinical_validity_table)

stability_table = pd.DataFrame([
    {
        "model": "SRM Global Linear",
        "mean_jaccard": selected_feature_jaccard(
            [fs for result in srm_interval_results.values() for fs in result["selected_features_by_fold"]]
        )["mean_jaccard"],
        "sign_stability": np.nan,
        "score_ranking_stability": np.nan,
    }
])
print("Feature robustness diagnostics")
display(stability_table)


Clinical validation of locked OOF composite scores


,analysis,clinical_variable,rho,p_value,n
0,cross_sectional,mfars_total,0.378729,2.732275e-08,202
1,longitudinal_delta,mfars_total,-0.078798,4.334648e-01,101
2,cross_sectional,sara_total,0.434407,1.051536e-10,202
3,longitudinal_delta,sara_total,-0.037669,7.084140e-01,101


Feature robustness diagnostics


,model,mean_jaccard,sign_stability,score_ranking_stability
0,SRM Global Linear,1.0,NaN,NaN


In [10]:
performance = assemble_performance_rows(
    "SRM Global Linear",
    composite_intervals=composite_intervals,
    clinical_intervals=clinical_intervals,
    single_feature_intervals=single_feature_intervals,
    clinical_validity=clinical_validity_table,
    stability=stability_table,
    cv_mode=f"subject-level grouped {CV_N_SPLITS}-fold",
    source="model_performance.ipynb",
)

log_models = best_model_rows_from_logs([
    REPO_ROOT / "results" / "all_model_optimization_log.csv",
    REPO_ROOT / "results" / "srm_composite_optimization_log.csv",
    REPO_ROOT / "results" / "comparator_optimization_log.csv",
    REPO_ROOT / "results" / "progression_dl_optimization_log.csv",
])
performance = append_log_model_summaries(performance, log_models)

performance_csv = save_one_performance_csv(performance, REPO_ROOT / "results" / "model_performance_summary.csv")
print(f"Saved one consolidated model-performance CSV: {performance_csv}")
display(performance)


Saved one consolidated model-performance CSV: /Users/robertwang/Documents/New_project/biomarkers/results/model_performance_summary.csv


,model,question,metric,role,value,n,status,evidence,cv_mode,source
0,SRM Global Linear,12-month sensitivity V1->V2,"V1->V2 paired d_z, CI, N, P(delta>0)",Primary,"1.2000891664038003 [1.001360894690141, 1.43443...",108.0,computed,composite V1->V2 OOF annual interval,subject-level grouped 5-fold,model_performance.ipynb
1,SRM Global Linear,12-month sensitivity V2->V3,"V2->V3 paired d_z, CI, N, P(delta>0)",Primary temporal replication,"0.42930710989742077 [0.2611439116843761, 0.640...",100.0,computed,composite V2->V3 OOF annual interval,subject-level grouped 5-fold,model_performance.ipynb
2,SRM Global Linear,24-month cumulative sensitivity,V1->V3 paired d_z,Secondary,1.326784,101.0,computed,composite V1->V3 cumulative,subject-level grouped 5-fold,model_performance.ipynb
3,SRM Global Linear,Direction consistency,P(delta > 0),Secondary,0.8796296296296297; 0.67,108.0,computed,annual V1->V2 and V2->V3 P(delta>0),subject-level grouped 5-fold,model_performance.ipynb
4,SRM Global Linear,Robustness,bootstrap CI for d_z,Primary uncertainty,"V1->V2 [1.001360894690141, 1.434437715764847];...",108.0,computed,annual interval bootstrap CI,subject-level grouped 5-fold,model_performance.ipynb
...,...,...,...,...,...,...,...,...,...,...
149,FusionMLP,Better than MRI alone?,vs strongest individual MRI feature,RQ1,NaN,NaN,not_available_in_existing_log,DL grouped-fold optimisation; clinical heads d...,group_kfold,all_model_optimization_log.csv
150,FusionMLP,Disease specific?,FRDA vs control change,Specificity,NaN,NaN,not_available_in_existing_log,DL grouped-fold optimisation; clinical heads d...,group_kfold,all_model_optimization_log.csv
151,FusionMLP,Clinically meaningful?,Spearman Z vs FARS/SARA,RQ3,NaN,NaN,not_available_in_existing_log,DL grouped-fold optimisation; clinical heads d...,group_kfold,all_model_optimization_log.csv
152,FusionMLP,Tracks clinical change?,Spearman delta Z vs delta FARS/SARA,Strong RQ3,NaN,NaN,not_available_in_existing_log,DL grouped-fold optimisation; clinical heads d...,group_kfold,all_model_optimization_log.csv
